# BNP Paribas- Bussines Case Project 2025/2026
**DATA VISUALIZATION AND PREPROCESSING NOTEBOOK**

**Group V**:
   - Alano Gonçalves (20250457)
   - Catarina Martins (20221914)
   - João Carichas (20250507)
   - Marta Ribeiro (20221886)
   - Nicole Nogueira(20221961)

# Index

- [1. Import](#import)
  - [1.1 Import libraries](#import-libraries)
  - [1.2 Import the dataset](#import-the-dataset)
- [2. Train and Test](#Train_and_Test)
- [3. Feature Selection](#Feature_Selection)
- [4. Model](#Model)

<div class="alert alert-block alert-info">

<a class="anchor" id="1. Import">    </a>
# 1. Import
       
</div>


[Back to Index](#index)

<a class="anchor" id="1.1 Import Libraries">

## 1.1 Import Libraries
    
</a>

In [1]:
#pip install lifelines
#!pip install scikit-survival pandas numpy scikit-learn
#import sys
#!{sys.executable} -m pip install scikit-survival
#!conda install -c conda-forge scikit-survival -y

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil
from suport import *
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler, StandardScaler
from lifelines import CoxPHFitter
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid
from sklearn.feature_selection import mutual_info_classif
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

# mostrar todas as colunas
pd.set_option('display.max_columns', None)

# mostrar toda a largura da tabela
pd.set_option('display.width', None)

# mostrar todo o conteúdo das células
pd.set_option('display.max_colwidth', None)

<a class="anchor" id="1.2 Import the Dataset">

## 1.2 Import the Dataset
    
</a>

In [3]:
#Alano
file_path = r"C:\Users\alano\Desktop\Mestrado\2- Semestre\Busines case\BNP\Projeto GIt\BNP-Paribas\Preprocessed_dataset_BNP_teste_alano.csv"
BNP = pd.read_csv(file_path)

In [4]:
date_cols = [
    "DCREAT",
    "DCRD_0",
    "OBS_END_DATE"
]

for col in date_cols:
    BNP[col] = pd.to_datetime(BNP[col], errors="coerce")


int32_cols = [
    "DURDEG",
    "RANGPRO_max",
    "RANGCLI_max",
    'delay_sum',
    "CSP"
]

for col in int32_cols:
    BNP[col] = BNP[col].astype("Int32")


float64_nullable_cols = [
    "RISKA_mean",
    "count_cl_mean",
    "subcount_total_mean"
]

for col in float64_nullable_cols:
    BNP[col] = BNP[col].astype("Float64")


object_cols = [
    'CONTRIB',
    "DOSSIER",
    "POLE",
    "PRODALP",
    "PAGAMENTO",
    "NATIO",
    "PTT",
    "MODCONTACTO",
    "kp_sqe",
    "is_risky",
    "sdem_SITFAM",
    "sdem_HABITAT"
]

for col in object_cols:
    BNP[col] = BNP[col].astype("object")


In [5]:
BNP.isna().sum()

CONTRIB                            0
DOSSIER                            0
is_san                             0
is_sol                             0
DCREAT                             0
DCRD_0                        107812
TIME_MONTHS                        0
DURDEG                             0
RANGPRO_max                        0
RANGCLI_max                        0
delay_sum                          0
MENSALIDADE                        0
CRD_min                            0
MTFINO                             0
RESSO_mean                         0
RISKA_mean                         0
NBENF_mean                        45
RN_mean                            0
SREC                               0
POLE                               0
PRODALP                            0
AGFIN                              0
PAGAMENTO                          0
CSP                                0
NATIO                              0
PTT                                0
MODCONTACTO                        0
C

In [6]:
BNP["NBENF_mean"] = BNP["NBENF_mean"].fillna(BNP["NBENF_mean"].median())

**METADATA:**
- DOSSIER
- is_san
- is_sol
- DCREAT
- DCRD_0
- CLOSE_DATE
- DURDEG
- RANGPRO-max
- RANGCLI_max
- delay-sum
- MENSALIDADE
- CRD_min
- MTFINO
- RESSO_mean
- RISKA_mean
- NBENF_mean
- RN_mean
- SREC
- PRODALP
- AGFIN
- PAGAMENTO
- CSP
- NATIO
- PTT
- MODCONTACTO
- NEXT_DOSSIER_DATE
- DAYS_TO_NEXT
- CHURN
- OBS_END_DATE
- DOSSIER_DURATION_DAYS
- N_PREVIOUS_DOSSIERS
- AVG_DURATION
- previous_churn_rate
- time_between_dossiers
- count_cl_mean
- montvenc_cl_mean
- montabatv_cl_mean
- dividas_cl_mean
- subcount_total_mean
- montenc_total_mean
- montabatv_totall_mean
- dividas_total_mean
- HIGH_RISK_CURRENT
- ks_score_tier
- is_risky
- ALLBD_mean_lifecycle_CL_N
- ALLBD_N_events_N
- sdem_SITFAM
- sdem_HABITAT
- sdem_age
- active_credit_ratio

In [7]:
BNP = BNP.drop(columns='TIME_MONTHS')


<a class="anchor" id="2. Train and Test Data">

## 2. Train and Test Data
    
</a>

In [8]:
train = BNP[BNP['CRD_min'] == 0].copy()
test  = BNP[BNP['CRD_min'] != 0].copy()

In [9]:
train.head()

,CONTRIB,DOSSIER,is_san,is_sol,DCREAT,DCRD_0,DURDEG,RANGPRO_max,RANGCLI_max,delay_sum,MENSALIDADE,CRD_min,MTFINO,RESSO_mean,RISKA_mean,NBENF_mean,RN_mean,SREC,POLE,PRODALP,AGFIN,PAGAMENTO,CSP,NATIO,PTT,MODCONTACTO,CHURN,time_to_churn,OBS_END_DATE,DOSSIER_DURATION_DAYS,N_PREVIOUS_DOSSIERS,AVG_DURATION,previous_churn_rate,time_between_dossiers,count_cl_mean,montvenc_cl_mean,montabatv_cl_mean,dividas_cl_mean,subcount_total_mean,montenc_total_mean,montabatv_total_mean,dividas_total_mean,HIGH_RISK_CURRENT,kp_sqe,ks_score_tier,is_risky,ALLBD_mean_duration_CL__N,ALLBD_mean_lifecycle_CL__N,ALLBD_N_events__N,sdem_SITFAM,sdem_HABITAT,sdem_age,active_credit_ratio
0,00008246f87bcc3c17b90629bb183fe2e58795176310f017217d7749af7ee981,36cc29a11081476fb7fcaed11aafef6fd441c3d172f4ffb8db787131bb069c76,1,0,2024-06-25,2024-09-30,84,3,19,1,158.359889,0.0,8000.00,1395.9935,0.0,0.0,0.0,0.0,P,EP,118.0,P,74,P,8600,W,1.0,17,2024-09-30,97,0,-1.0,-1.0,-1.0,0.36,0.0,0.0,2565.7936,5.28,0.0,0.0,17196.2076,0.0,H,2.0,True,120.0,0.033333,10.0,S,F,33.0,0.272727
1,00008246f87bcc3c17b90629bb183fe2e58795176310f017217d7749af7ee981,a34f4d89a43b924b483dc829a65ccba8b0f90e0afc206396405115a02cd513fc,1,0,2024-09-13,2025-11-30,120,13,19,1,280.401466,0.0,16000.00,1513.4660,0.0,0.0,0.0,0.0,P,EP,118.0,P,80,P,8600,W,1.0,17,2025-11-30,443,1,97.0,-1.0,80.0,0.36,0.0,0.0,2565.7936,5.28,0.0,0.0,17196.2076,0.0,H,2.0,True,120.0,0.033333,10.0,S,F,33.0,0.272727
3,0000c74654405ec1da4dbdcd00b86e397954043965d98e542d19fa4808c6b65a,796b87efb05b1ec3f822189b7b5fe25be06390e7462d228bc34ef083de72f011,0,1,2001-09-21,2024-03-31,60,21,21,0,424.699203,0.0,13467.54,678.4830,0.0,2.0,1.0,0.0,P,EP,117.0,P,60,P,2745,T,1.0,270,2024-03-31,8227,0,-1.0,-1.0,-1.0,1.32,0.0,0.0,7120.2352,5.2,0.0,0.0,26643.3160,0.0,Unknown,1.0,1.0,60.0,0.178571,8.0,C,A,44.0,0.166667
4,0000c74654405ec1da4dbdcd00b86e397954043965d98e542d19fa4808c6b65a,1c848abbdf2867f690d6c07cd30f110a0b49fcd44b7202d95cfd8ba504a84f6c,0,1,2001-09-27,2024-03-31,60,21,21,0,184.214593,0.0,5985.57,678.4830,0.0,2.0,1.0,0.0,P,EP,117.0,P,60,P,2745,T,1.0,270,2024-03-31,8221,1,8227.0,-1.0,6.0,1.32,0.0,0.0,7120.2352,5.2,0.0,0.0,26643.3160,0.0,Unknown,1.0,1.0,60.0,0.178571,8.0,C,A,44.0,0.166667
5,0000e359b15a80ba12c7f60c0fe06ffc68621c87f13a4f88f54188543d9a09c8,b83e6ef0c9c12f8304d86c40458a5b88ec72e534201c32a1a99618e5be83ced1,1,0,2019-01-28,2024-01-31,72,34,34,0,56.017772,0.0,2500.00,838.1860,0.0,0.0,1.0,0.0,P,EP,120.0,N,91,P,2855,A,1.0,60,2024-01-31,1829,0,-1.0,-1.0,-1.0,1.32,0.0,0.0,7120.2352,5.2,0.0,0.0,26643.3160,0.0,Unknown,1.0,1.0,0.0,0.000000,3.0,S,A,69.0,0.000000


In [10]:
train['CHURN'].value_counts()

CHURN
1.0    75857
0.0     1931
Name: count, dtype: int64

In [11]:
test.head()

,CONTRIB,DOSSIER,is_san,is_sol,DCREAT,DCRD_0,DURDEG,RANGPRO_max,RANGCLI_max,delay_sum,MENSALIDADE,CRD_min,MTFINO,RESSO_mean,RISKA_mean,NBENF_mean,RN_mean,SREC,POLE,PRODALP,AGFIN,PAGAMENTO,CSP,NATIO,PTT,MODCONTACTO,CHURN,time_to_churn,OBS_END_DATE,DOSSIER_DURATION_DAYS,N_PREVIOUS_DOSSIERS,AVG_DURATION,previous_churn_rate,time_between_dossiers,count_cl_mean,montvenc_cl_mean,montabatv_cl_mean,dividas_cl_mean,subcount_total_mean,montenc_total_mean,montabatv_total_mean,dividas_total_mean,HIGH_RISK_CURRENT,kp_sqe,ks_score_tier,is_risky,ALLBD_mean_duration_CL__N,ALLBD_mean_lifecycle_CL__N,ALLBD_N_events__N,sdem_SITFAM,sdem_HABITAT,sdem_age,active_credit_ratio
2,0000ab2116257783438c70ff85a3e98f2d4194ebe534349a33373dfcb3a3a297,9e4d186f6f66f2da4b816cbc6f6e05e640caf5ded07956d5fde4919118d37d43,0,0,2018-03-29,NaT,120,91,91,0,347.447280,8115.247,20000.0,1113.258,0.0,1.0,0.0,15.106391,P,EP,120.0,P,80,P,2845,A,NaN,92,2025-11-30,2803,0,-1.0,-1.0,-1.0,4.32,0.00000,0.0,26448.032000,9.72,0.000000,0.0,34000.14920,0.0,D,1.0,False,120.0,0.675000,10.0,C,P,52.0,0.090909
6,0000f858346061c53064586a3347b34659565a6712d004e64309c2473f76faed,29c3cfb34c4e2ecd6749b0c1a6205dfce33c0e717514560542d65af5e63e2157,0,0,2019-09-23,NaT,84,74,74,0,100.073575,883.500,5000.0,1314.144,0.0,2.0,0.0,0.000000,P,EP,118.0,P,80,P,2635,W,NaN,74,2025-11-30,2260,0,-1.0,-1.0,-1.0,1.04,0.00000,0.0,2082.786000,8.32,0.000000,0.0,93756.88440,0.0,Unknown,1.0,1.0,84.0,0.761905,4.0,U,A,39.0,0.400000
7,00025459b703e1c308553e83a6d545a71fe6a787c2dd1c62d26faa0207cc5b08,446d96905d26356fb4f1d3e1e6da3868b8106b09a558e61d70dcb3944412ecac,0,0,2023-01-09,NaT,60,35,37,0,162.097525,3419.266,6000.0,1031.650,0.0,0.0,0.0,0.000000,P,EPF,117.0,P,80,P,5090,A,NaN,34,2025-11-30,1056,0,-1.0,-1.0,-1.0,3.0,0.00000,0.0,22980.086400,4.28,0.000000,0.0,23082.37120,0.0,Unknown,1.0,1.0,60.0,0.416667,17.0,C,P,55.0,0.111111
9,00041ebafb1270a818c30cb1fb20d3699002196644ea8fd9df9425df6f49db00,26d59ea8a93be739e9fc4a0404f07c73aae276243e5cd6236580eb19aa138a56,0,0,2021-02-10,NaT,84,43,43,2,463.755537,14665.602,20500.0,1113.258,0.3,0.0,1.9,-46.375600,N,EXT,119.0,P,80,P,2835,A,NaN,46,2025-11-30,1754,0,-1.0,-1.0,-1.0,9.75,2416.34125,0.0,55289.663333,13.625,4038.840833,0.0,60154.73875,1.0,F,3.0,True,84.0,0.523810,13.0,C,A,35.0,0.071429
11,00050fe9f0e69ce221a574af0baaff0b37c598af7a5cc6eb42a2afd52d4e0d4a,cd128e1f3c6d6bc9dd2414eeb60f44241977a35fd94885881bfec649d1b8bace,0,0,2023-09-11,NaT,48,27,121,0,131.155715,2349.858,4080.0,877.293,0.0,1.0,0.0,0.000000,P,EP,117.0,P,80,P,4475,A,NaN,26,2025-11-30,811,0,-1.0,-1.0,-1.0,1.0,0.00000,0.0,2987.307200,2.8,0.000000,0.0,3183.04880,0.0,A,1.0,False,48.0,0.354167,12.0,C,P,54.0,0.153846


In [12]:
test['CHURN'].value_counts()

Series([], Name: count, dtype: int64)

In [13]:
duration = np.where(
    train['CHURN'] == 1,
    (train['DCRD_0'] - train['DCREAT']).dt.days + 30,
    (train['DCRD_0'] - train['DCREAT']).dt.days
)

train['time_to_churn'] = pd.Series(duration, index=train.index).clip(lower=0)

<a class="anchor" id="2.1 TRAIN TEST SPLIT">

## 2.1 TRAIN TEST SPLIT
    
</a>

In [14]:
X = train.drop(columns=['CHURN'])
y = train['CHURN']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y)

In [15]:
X_train

,CONTRIB,DOSSIER,is_san,is_sol,DCREAT,DCRD_0,DURDEG,RANGPRO_max,RANGCLI_max,delay_sum,MENSALIDADE,CRD_min,MTFINO,RESSO_mean,RISKA_mean,NBENF_mean,RN_mean,SREC,POLE,PRODALP,AGFIN,PAGAMENTO,CSP,NATIO,PTT,MODCONTACTO,time_to_churn,OBS_END_DATE,DOSSIER_DURATION_DAYS,N_PREVIOUS_DOSSIERS,AVG_DURATION,previous_churn_rate,time_between_dossiers,count_cl_mean,montvenc_cl_mean,montabatv_cl_mean,dividas_cl_mean,subcount_total_mean,montenc_total_mean,montabatv_total_mean,dividas_total_mean,HIGH_RISK_CURRENT,kp_sqe,ks_score_tier,is_risky,ALLBD_mean_duration_CL__N,ALLBD_mean_lifecycle_CL__N,ALLBD_N_events__N,sdem_SITFAM,sdem_HABITAT,sdem_age,active_credit_ratio
48063,4277ae9bef265530b3a7f0010f198c32a908c514904c3cbb7b217945da668507,f3014c55c23851918ceb4d6a6f576c3846270f104369f649b54b50cd81651c32,1,0,2022-06-07,2025-01-31,84,31,31,1,229.229218,0.0,10400.00,452.938000,0.0,0.000000,0.000000,-35.266000,P,EPF,119.0,P,58,P,4400,A,999,2025-01-31,969,0,-1.0,-1.0,-1.0,1.0,5.5864,0.0,7674.389200,1.64,5.5864,0.0,8311.714800,1.0,Unknown,1.0,1.0,84.0,0.011905,11.0,C,P,52.0,0.166667
124179,ab6583343d71465306cb78102f0f7944d3e87425ca11eb24f6e9b9976a217009,f35044c2f66513341fc83a5900a21cba398add0d4f4ee13629ff1aca85d331ce,0,1,2011-04-28,2024-03-31,96,22,22,0,262.394397,0.0,12500.00,815.294000,0.0,0.000000,0.000000,0.000000,P,EP,119.0,N,80,P,4540,W,4751,2024-03-31,4721,0,-1.0,-1.0,-1.0,1.32,0.0000,0.0,7120.235200,5.2,0.0000,0.0,26643.316000,0.0,Unknown,1.0,1.0,0.0,0.000000,3.0,C,P,59.0,0.000000
110903,99514aeadae8b29f8b656fa14062374666e0b5ca7329253719b31a8381bfbf74,fec6706ea221165044d24aadbf7238c5cc01a783245ec29dfb295a75127cd468,1,0,2023-05-07,2024-02-29,96,9,9,1,99.567126,0.0,5000.00,1118.707000,0.0,1.000000,0.000000,0.000000,P,EP,120.0,P,31,P,2855,W,298,2024-02-29,298,0,-1.0,-1.0,-1.0,0.166667,0.0000,0.0,2349.679167,4.458333,0.0000,0.0,142807.957500,0.0,Unknown,1.0,1.0,120.0,0.083333,6.0,U,F,41.0,0.142857
119942,a5a44f4b0bcf3019e34916fff5690ddc886b926be648f9a72f2bb5072be4a447,ffa2044c74bee08f4160729aef30219a7d820053d20d03bfa00338134750891b,0,1,2020-06-23,2024-01-31,40,40,40,0,325.849190,0.0,8955.45,2595.150000,0.0,1.000000,0.000000,0.000000,P,EPF,118.0,P,33,P,4415,W,1347,2024-01-31,1317,0,-1.0,-1.0,-1.0,5.0,0.0000,0.0,27491.790000,11.0,0.0000,0.0,121092.450000,0.0,Unknown,1.0,True,0.0,0.000000,5.0,C,A,51.0,0.000000
18074,192c0c90e77dbe9d298650cfb0eb03f122c62276981521f55dfe0e8e2cd3c4d9,b11a7256242b6bbcd1e8d9722c89814a96242dc28d9f0015810dc30e7a4a99b5,1,0,2023-04-27,2024-12-31,64,19,19,1,84.679882,0.0,3500.00,1490.138143,1.0,0.285714,0.285714,-9.938714,P,EP,120.0,P,31,P,2100,W,644,2024-12-31,614,0,-1.0,-1.0,-1.0,2.44,597.0520,0.0,23045.359600,6.48,777.0168,0.0,120373.034800,1.0,Unknown,1.0,1.0,84.0,0.011905,5.0,X,A,42.0,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124460,abd4672a3d0f3d620f18df4f057a4de9315f3c771b33b4daf2d076523b33f038,6dd90bb5b4698eb4b81a50957e6c1d8d76b6bf89ecadb399a52b4f874bc29423,1,0,2017-09-19,2024-01-31,84,58,58,0,117.216763,0.0,5500.00,721.299000,0.0,1.000000,1.000000,0.000000,P,EXT,119.0,P,80,P,4820,W,2355,2024-01-31,2325,0,-1.0,-1.0,-1.0,1.32,0.0000,0.0,7120.235200,5.2,0.0000,0.0,26643.316000,0.0,Unknown,1.0,1.0,0.0,0.000000,6.0,C,F,43.0,0.000000
1917,02b8411f41af311e44f46e0081609e83c2d89cefd24f5b0cd693a89250c40b50,524fdf553056787bf930dd04e5b1ded003e54aadfd854dc124495d1fc9ff37e7,1,0,2021-06-09,2024-09-30,84,39,39,1,63.304087,0.0,3000.00,2569.826000,0.0,0.000000,0.000000,0.000000,P,EP,118.0,P,74,P,9325,W,1239,2024-09-30,1209,0,-1.0,-1.0,-1.0,1.153846,0.0000,0.0,3289.668462,8.153846,0.0000,0.0,89914.540769,0.0,B,1.0,False,0.0,0.000000,11.0,S,P,35.0,0.000000
13340,1279515fc45d6cb103aeb587900be3e8eec51478bfdeff340171bcff15e6908f,9a8b267e347ad793e4550804d72f20893146dcede4b82c992ebc7f0152ed7006,1,0,2024-04-08,2025-04-30,84,9,20,1,105.543186,0.0,6000.00,1394.308500,0.0,0.00

<a class="anchor" id="2.2. CROSS VALIDATION">

## 2.2 CROSS VALIDATION
    
</a>

TimeSeriesSplit

Só usaria se quiseres simular previsão no tempo.

Porquê:

respeita a ordem temporal

treinas no passado e validas no futuro

aproxima melhor o cenário real de produção

Usaria quando:

queres prever churn de contratos futuros com base em contratos passados

a data (DCREAT, DCRD_0, OBS_END_DATE) for central no problema

quiseres evitar misturar passado e futuro

In [16]:
train = train.sort_values('DCRD_0').reset_index(drop=True)

X = train.drop(columns=['CHURN'])
y = train['CHURN']

tscv = TimeSeriesSplit(n_splits=5)

In [17]:
for fold, (train_idx, val_idx) in enumerate(tscv.split(X), 1):
    X_train_cv, X_val_cv = X.iloc[train_idx], X.iloc[val_idx]
    y_train_cv, y_val_cv = y.iloc[train_idx], y.iloc[val_idx]

    print(f"Fold {fold}")
    print("Train:", X_train_cv.shape, "Validation:", X_val_cv.shape)

Fold 1
Train: (12968, 52) Validation: (12964, 52)
Fold 2
Train: (25932, 52) Validation: (12964, 52)
Fold 3
Train: (38896, 52) Validation: (12964, 52)
Fold 4
Train: (51860, 52) Validation: (12964, 52)
Fold 5
Train: (64824, 52) Validation: (12964, 52)


<a class="anchor" id="3. FEATURE SELCTION">

## 3. FEATURE SELCTION
    
</a>

In [18]:
# colunas que NÃO entram no feature selection
excluded_cols = [
    'CHURN',            # target / event
    'time_to_churn',    # duration
    'CONTRIB',          # id cliente
    'DOSSIER',          # id dossier
    'DCREAT',           # data
    'DCRD_0',           # data
    'OBS_END_DATE'      # data de observação
]

# dataset só com colunas elegíveis para feature selection
train_fs = train.drop(columns=excluded_cols, errors='ignore')

# separar numéricas e categóricas
numeric_cols = train_fs.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = train_fs.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

In [19]:
train_fs_num = train_fs[numeric_cols].copy()
train_fs_cat = train_fs[categorical_cols].copy()

## Encoding 

In [20]:
cardinality = train_fs_cat.nunique(dropna=False).sort_values()

low_card_cols = cardinality[cardinality <= 10].index.tolist()
high_card_cols = cardinality[cardinality > 10].index.tolist()

print("Baixa cardinalidade:", low_card_cols)
print("Alta cardinalidade:", high_card_cols)

Baixa cardinalidade: ['POLE', 'PAGAMENTO', 'is_risky', 'PRODALP', 'NATIO', 'MODCONTACTO', 'sdem_HABITAT', 'sdem_SITFAM']
Alta cardinalidade: ['kp_sqe', 'PTT']


In [21]:
def group_rare_categories(series, min_freq=0.01):
    s = series.astype('object').fillna('MISSING').copy()
    freq = s.value_counts(normalize=True)
    rare_cats = freq[freq < min_freq].index
    s = s.where(~s.isin(rare_cats), 'OTHER')
    return s

for col in low_card_cols:
    X[col] = group_rare_categories(X[col], min_freq=0.01)

for col in high_card_cols:
    s = X[col].astype('object').fillna('MISSING')
    freq_map = s.value_counts(normalize=True)
    X[col + '_freq'] = s.map(freq_map)

train_fs_cat = pd.get_dummies(train_fs_cat, columns=low_card_cols, drop_first=True, dummy_na=False)
train_fs_cat = train_fs_cat.drop(columns=high_card_cols)

train_fs_cat = train_fs_cat.replace([np.inf, -np.inf], np.nan)

for col in train_fs_cat.columns:
    if train_fs_cat[col].dtype == 'bool':
        train_fs_cat[col] = train_fs_cat[col].astype(int)

print("Shape final:", train_fs_cat.shape)
print(train_fs_cat.head())

Shape final: (77788, 30)
   POLE_P  PAGAMENTO_N  PAGAMENTO_P  is_risky_False  is_risky_True  \
0       1            0            1               0              0   
1       1            0            1               0              0   
2       1            0            1               0              0   
3       1            0            1               0              0   
4       1            0            1               0              0   

   PRODALP_EP  PRODALP_EPF  PRODALP_EXT  NATIO_D  NATIO_P  NATIO_X  \
0           1            0            0        0        1        0   
1           1            0            0        0        1        0   
2           1            0            0        0        1        0   
3           0            1            0        0        1        0   
4           1            0            0        0        1        0   

   MODCONTACTO_C  MODCONTACTO_T  MODCONTACTO_V  MODCONTACTO_W  MODCONTACTO_X  \
0              0              0              0       

C:\Users\alano\AppData\Local\Temp\ipykernel_24140\2127405850.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = X[col].astype('object').fillna('MISSING')


SCALING

In [22]:
scaler = RobustScaler()
train_fs_num = pd.DataFrame(
    scaler.fit_transform(train_fs_num),
    columns=train_fs_num.columns,
    index=train_fs_num.index)

In [23]:
train_scaled = pd.concat([train_fs_num, train_fs_cat], axis=1)

In [24]:
corr_matrix = train_scaled.corr().abs()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Colunas para remover:", to_drop)
print("Quantidade:", len(to_drop))

Colunas para remover: ['is_sol', 'PAGAMENTO_P']
Quantidade: 2


In [25]:
train_scaled = train_scaled.drop(columns=to_drop)
train_scaled

,is_san,DURDEG,RANGPRO_max,RANGCLI_max,delay_sum,MENSALIDADE,CRD_min,MTFINO,RESSO_mean,RISKA_mean,NBENF_mean,RN_mean,SREC,AGFIN,CSP,DOSSIER_DURATION_DAYS,N_PREVIOUS_DOSSIERS,AVG_DURATION,previous_churn_rate,time_between_dossiers,count_cl_mean,montvenc_cl_mean,montabatv_cl_mean,dividas_cl_mean,subcount_total_mean,montenc_total_mean,montabatv_total_mean,dividas_total_mean,HIGH_RISK_CURRENT,ks_score_tier,ALLBD_mean_duration_CL__N,ALLBD_mean_lifecycle_CL__N,ALLBD_N_events__N,sdem_age,active_credit_ratio,POLE_P,PAGAMENTO_N,is_risky_False,is_risky_True,PRODALP_EP,PRODALP_EPF,PRODALP_EXT,NATIO_D,NATIO_P,NATIO_X,MODCONTACTO_C,MODCONTACTO_T,MODCONTACTO_V,MODCONTACTO_W,MODCONTACTO_X,sdem_HABITAT_A,sdem_HABITAT_E,sdem_HABITAT_F,sdem_HABITAT_L,sdem_HABITAT_O,sdem_HABITAT_P,sdem_HABITAT_X,sdem_SITFAM_D,sdem_SITFAM_F,sdem_SITFAM_P,sdem_SITFAM_S,sdem_SITFAM_U,sdem_SITFAM_V,sdem_SITFAM_X
0,-1.0,-0.2500,0.902778,0.567568,0.0,-0.053140,0.0,0.071429,1.629096,0.0,0.0,0.000000,0.000,0.0,1.24,0.624383,0.0,0.0,0.0,0.0,-1.32,0.0000,0.0,-0.522114,-0.346667,0.0000,0.0,-0.463941,0.0,0.0,-0.309524,-0.148423,-0.428571,1.294118,0.180000,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,-1.0,0.0625,1.319444,0.972973,0.0,1.573854,0.0,1.428571,-0.462297,0.0,1.0,0.000000,0.000,2.0,0.80,0.857646,0.0,0.0,0.0,0.0,0.18,0.0000,0.0,-0.489770,-1.233333,0.0000,0.0,-0.459081,0.0,0.0,-0.309524,-0.148423,-0.857143,-0.470588,-0.720000,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
2,0.0,-0.5000,0.430556,0.108108,0.0,-0.522744,0.0,-0.357143,0.153342,0.0,0.0,0.000000,0.000,0.0,-1.00,0.305144,0.0,0.0,0.0,0.0,0.48,0.0000,0.0,0.711619,1.200000,0.0000,0.0,0.676936,0.0,0.0,0.690476,0.519481,0.142857,0.294118,0.900000,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0
3,0.0,0.2500,-0.402778,0.270270,0.0,6.160916,0.0,6.814286,0.498691,0.0,0.0,2.000000,0.000,0.0,0.40,-0.382664,0.0,0.0,0.0,0.0,0.68,0.0000,0.0,3.016234,-0.733333,678.2328,0.0,0.667225,1.0,0.0,1.119048,0.215213,2.857143,0.529412,-0.533793,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
4,-1.0,-0.8750,0.069444,-0.243243,0.0,-0.252068,0.0,-0.357143,-0.395600,0.0,0.0,1.000000,0.000,0.0,0.00,-0.085271,0.0,0.0,0.0,0.0,-1.32,0.0000,0.0,-0.522114,-1.040000,0.0000,0.0,-0.463018,0.0,0.0,-0.309524,-0.148423,-0.714286,0.058824,1.980000,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77783,-1.0,-1.0000,-0.097222,-0.405405,2.0,-0.059988,0.0,-0.357143,-0.095550,0.0,0.0,0.777778,-23.321,2.0,0.80,-0.175476,0.0,0.0,0.0,0.0,0.56,129.8964,0.0,-0.113008,-0.440000,207.9828,0.0,-0.330652,1.0,0.0,-0.023810,1.929499,-0.285714,0.352941,0.822857,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0
77784,0.0,-1.0000,-0.597222,-0.891892,1.0,1.009110,0.0,0.000000,-1.359131,0.0,0.0,0.000000,0.000,2.0,0.80,-0.599718,0.0,0.0,0.0,0.0,-0.32,0.0000,0.0,-0.106244,-1.400000,0.0000,0.0,-0.367443,0.0,2.0,0.404762,0.408163,0.000000,0.000000,0.180000,1,0,0,1,1,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
77785,0.0,-1.0000,-0.458333,1.243243,1.0,0.065432,0.0,-0.285714,3.678281,0.0,0.0,0.000000,0.000,2.0,-0.40,-0.495419,0.0,0.0,0.0,0.0,-0.32,0.0000,0.0,-0.313442,0.266667,0.0000,0.0,-0.287165,0.0,0.0,-0.023810,0.111317,0.714286,1.588235,0.051429,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
77786,-1.0,-1.0000,-0.097222,1.675676,0.0,-0.259964,0.0,-0.428571,1.187349,0.0,0.0,0.000000,0.000,0.0,0.00,-0.229739,2.0,2401.0,0.0,978.0,1.20,0.0000,0.0,-0.151688,0.920000,0.0000,0.0,0.112532,0.0,0.0,0.610119,1.275819,1.000000,-0.235294,1.305000,1,0,0,1,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0


## Mutual Information (MI)

We use Mutual Information (MI) as an initial feature selection step because it measures how much information each feature provides about the target variable, in this case churn. Unlike simple linear correlation, MI can capture both linear and non-linear relationships, which is useful when customer behavior is complex and not strictly linear. This makes it a strong first filter for identifying potentially relevant variables.

In this project, MI is especially helpful because the dataset contains many variables, including encoded categorical features that can greatly increase dimensionality. Applying MI before the survival model reduces noise, removes weak predictors, and lowers computational cost. This allows the later modeling stage to focus on the most informative features while keeping the process more efficient and stable.

In [26]:
X=train_scaled
Y=train['CHURN']

In [27]:
mi = mutual_info_classif(X, y, random_state=42)

mi_df = pd.DataFrame({
    'feature': X.columns,
    'mi_score': mi
}).sort_values('mi_score', ascending=False)

print(mi_df.head(20))

                       feature  mi_score
35                      POLE_P  0.022135
31  ALLBD_mean_lifecycle_CL__N  0.020928
43                     NATIO_P  0.017765
48               MODCONTACTO_W  0.014024
39                  PRODALP_EP  0.013790
30   ALLBD_mean_duration_CL__N  0.012735
23             dividas_cl_mean  0.008405
0                       is_san  0.008200
4                    delay_sum  0.007592
27          dividas_total_mean  0.007279
20               count_cl_mean  0.006505
34         active_credit_ratio  0.005902
50              sdem_HABITAT_A  0.005818
24         subcount_total_mean  0.005063
15       DOSSIER_DURATION_DAYS  0.004794
10                  NBENF_mean  0.004567
8                   RESSO_mean  0.003847
52              sdem_HABITAT_F  0.003741
55              sdem_HABITAT_P  0.003071
2                  RANGPRO_max  0.002895


In [28]:
selected_features = mi_df[mi_df['mi_score'] > 0.001]['feature'].tolist()
X_selected = X[selected_features].copy()

print("Nº features selecionadas:", len(selected_features))

Nº features selecionadas: 38


<a class="anchor" id="4. MODEL">

## 4. MODEL
    
</a>

In [29]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [5, 10, 20],
    "min_samples_leaf": [5, 10, 15, 20],
    "max_features": ["sqrt", "log2", 0.3, 0.5]
}

In [30]:
# target survival
y_surv = Surv.from_dataframe(
    event='CHURN',
    time='time_to_churn',
    data=train
)

# garantir que as linhas batem certo
X_rsf = X_selected.copy()

# modelo
rsf = RandomSurvivalForest(
    n_estimators=50,
    min_samples_split=20,
    min_samples_leaf=50,
    max_depth=10,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42,
    low_memory=True)

# treino
rsf.fit(X_rsf, y_surv)

print("Modelo RSF treinado com sucesso.")

Modelo RSF treinado com sucesso.


In [31]:
print("C-index:", rsf.score(X_rsf, y_surv))

C-index: 0.797103745431613


In [ ]:
# garantir que o train está ordenado no tempo
train = train.sort_values("DCRD_0").reset_index(drop=True)

# garantir alinhamento entre features e train
X_model = X_selected.loc[train.index].copy()

# target survival
y_surv = Surv.from_dataframe(
    event="CHURN",
    time="time_to_churn",
    data=train
)

# TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

# grelha de hiperparâmetros
param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [5, 10, None],
    "min_samples_split": [10, 20, 30],
    "min_samples_leaf": [10, 20, 50],
    "max_features": ["sqrt", "log2", 0.3]
}

results = []

for params in ParameterGrid(param_grid):
    fold_scores = []

    print(f"\nTesting params: {params}")

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_model), 1):
        X_train_cv = X_model.iloc[train_idx]
        X_val_cv = X_model.iloc[val_idx]

        y_train_cv = y_surv[train_idx]
        y_val_cv = y_surv[val_idx]

        rsf = RandomSurvivalForest(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            n_jobs=-1,
            random_state=42,
            low_memory=True
        )

        rsf.fit(X_train_cv, y_train_cv)
        c_index = rsf.score(X_val_cv, y_val_cv)
        fold_scores.append(c_index)

        print(f"  Fold {fold} C-index: {c_index:.4f}")

    mean_cindex = np.mean(fold_scores)
    std_cindex = np.std(fold_scores)

    results.append({
        **params,
        "mean_cindex": mean_cindex,
        "std_cindex": std_cindex
    })

    print(f"  Mean C-index: {mean_cindex:.4f}")
    print(f"  Std C-index: {std_cindex:.4f}")

# resultados finais ordenados
results_df = pd.DataFrame(results).sort_values("mean_cindex", ascending=False).reset_index(drop=True)

print("\nTop 10 combinações:")
print(results_df.head(10))


Testing params: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 10, 'n_estimators': 50}
  Fold 1 C-index: 0.5044
  Fold 2 C-index: 0.5290
  Fold 3 C-index: 0.4979
  Fold 4 C-index: 0.4960
  Fold 5 C-index: 0.5030
  Mean C-index: 0.5061
  Std C-index: 0.0119

Testing params: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 10, 'n_estimators': 100}
  Fold 1 C-index: 0.5153
  Fold 2 C-index: 0.5277
  Fold 3 C-index: 0.4963
  Fold 4 C-index: 0.4974
  Fold 5 C-index: 0.5032
  Mean C-index: 0.5080
  Std C-index: 0.0119

Testing params: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 10, 'n_estimators': 150}
  Fold 1 C-index: 0.5196
  Fold 2 C-index: 0.5271
  Fold 3 C-index: 0.4972
  Fold 4 C-index: 0.4962
  Fold 5 C-index: 0.5030
  Mean C-index: 0.5086
  Std C-index: 0.0125

Testing params: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 20, 